# Etapa 01 - Denoising de ECG

Essa implementação foi focada na Etapa 01 descrita no documento forcenido pelo prof. Ferrari, em
[`docs/Projeto_02_Etapa_01.pdf`](../docs/Projeto_02_Etapa_01.pdf).

Aplicamos quatro métodos de filtragem digital ao mesmo segmento de ECG real, na sequencia
**HP (BW) → BS (PLI) → LP (EMG)**:

1. FIR pelo método da janela (Hamming);
2. FIR por Parks-McClellan (equiripple);
3. FIR baseado em FFT (mesma resposta, convolução rápida);
4. IIR Butterworth.

A avaliação foi feita por meio de redução em dB na banda alvo,
preservação na banda passante, estabilidade temporal de picos R, coerência
filtro/PSD e linearidade de fase.


## 1. Setup e parâmetros

In [ ]:
%load_ext autoreload
%autoreload 2

from __future__ import annotations

import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal as sp_signal
import wfdb

from src.config import mitdb_record_dir, PROCESSED_DATA_DIR
from src.preprocessing import fir_filters as fir
from src.preprocessing import iir_filters as iir
from src.preprocessing import metrics as M
from src.preprocessing import gabor_filters as gab
from src.visualization.ecg_plots import plot_overlay, plot_psd, plot_filter_response

warnings.filterwarnings("ignore", category=RuntimeWarning)
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (11, 3.4)})

DATA_DIR = mitdb_record_dir()

### 1.1. Parâmetros globais

`SAMPLE_DURATION_SEC` controla o tamanho do segmento. Demais parâmetros
seguem a Seção 4.3 do PDF (cortes em 0,5 Hz e 40 Hz; notch em 60 Hz).

In [ ]:
SAMPLE_DURATION_SEC: float = 10.0

SAMPLE_START_SEC: float = 0.0 

# registro e canal 100
RECORD_ID: str = "100"
CHANNEL: int = 0

# freq usada pelos filtros (Hz)
HP_CUT_HZ: float = 0.5
HP_TRANS_HZ: float = 1.0
PLI_LO_HZ: float = 59.0
PLI_HI_HZ: float = 61.0
PLI_TRANS_HZ: float = 1.0
LP_CUT_HZ: float = 40.0
LP_TRANS_HZ: float = 8.0

# bandas para as metricas
PASSBAND: tuple[float, float] = (1.0, 30.0)
BAND_BW: tuple[float, float] = (0.0, 0.5)
BAND_PLI: tuple[float, float] = (58.0, 62.0)
BAND_EMG: tuple[float, float] = (40.0, 90.0)

## 2. Carregamento do segmento

Recortamos um trecho contínuo de `SAMPLE_DURATION_SEC` segundos do registo
100 da MIT-BIH (batimentos normais sinusais), começando em
`SAMPLE_START_SEC`. As anotações WFDB de pico R dentro da janela são
preservadas apenas para a métrica de estabilidade temporal.

In [ ]:
rec = wfdb.rdrecord((DATA_DIR / RECORD_ID).as_posix())
ann = wfdb.rdann((DATA_DIR / RECORD_ID).as_posix(), "atr")
x_full = rec.p_signal[:, CHANNEL]
FS = float(rec.fs)

start_sample = int(round(SAMPLE_START_SEC * FS))
end_sample = start_sample + int(round(SAMPLE_DURATION_SEC * FS))
end_sample = min(end_sample, len(x_full))

x = np.asarray(x_full[start_sample:end_sample], dtype=float)
t = np.arange(len(x)) / FS

# anotacoes WFDB que caem dentro da janela (em amostras locais)
mask = (ann.sample >= start_sample) & (ann.sample < end_sample)
ANN_LOCAL = ann.sample[mask] - start_sample

print(f"fs = {FS:g} Hz | duracao = {SAMPLE_DURATION_SEC:g} s | amostras = {len(x)}")
print(f"registo {RECORD_ID} | inicio = {SAMPLE_START_SEC:g} s | anotacoes na janela = {len(ANN_LOCAL)}")

fig, ax = plt.subplots(figsize=(11, 2.8))
ax.plot(t, x, color="#1f77b4", lw=0.8)
ax.scatter(ANN_LOCAL / FS, np.full(len(ANN_LOCAL), x.max() * 1.05),
           marker="v", color="#d62728", s=18, label="picos R (anotacoes)")
ax.set_title(f"Sinal cru - rec {RECORD_ID} (Normal sinusal)")
ax.set_xlabel("tempo (s)"); ax.set_ylabel("mV"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 3. Projeto dos filtros

Cada método projeta três filtros (HP em 0,5 Hz, BS em 59 Hz a 61 Hz, LP em
40 Hz). Para FIR usamos a relação `N ≈ 3,3 · fs / Δf`. Para IIR usamos
ordem 4 em cada estágio.

### 3.1. Método A: FIR pelo método da janela (Hamming)

In [ ]:
hA_hp = fir.design_highpass(FS, HP_CUT_HZ, transition_hz=HP_TRANS_HZ, window="hamming")
hA_bs = fir.design_bandstop(FS, PLI_LO_HZ, PLI_HI_HZ, transition_hz=PLI_TRANS_HZ, window="hamming")
hA_lp = fir.design_lowpass(FS, LP_CUT_HZ, transition_hz=LP_TRANS_HZ, window="hamming")
print(f"FIR janela | HP N={len(hA_hp)} | BS N={len(hA_bs)} | LP N={len(hA_lp)}")

### 3.2. Método B: FIR por Parks-McClellan (equiripple)

`scipy.signal.remez` resolve o problema minimax e produz coeficientes com
ondulação uniforme nas bandas. Equiripple atinge a especificação com `N`
menor que o método da janela; mantemos a comparação na **mesma especificação
de bandas**, deixando que cada método escolha o `N` apropriado.

In [ ]:
# Parks-McClellan (equiripple) usa N moderado (~fs/Df, limitado a 801).
# Ordens muito altas com bandas proximas de 0 Hz impedem a convergencia do
# algoritmo de Remez; equiripple ja atinge a especificacao com menos taps.
hB_hp = fir.design_highpass_remez(FS, HP_CUT_HZ, transition_hz=HP_TRANS_HZ)
hB_bs = fir.design_bandstop_remez(FS, PLI_LO_HZ, PLI_HI_HZ, transition_hz=PLI_TRANS_HZ)
hB_lp = fir.design_lowpass_remez(FS, LP_CUT_HZ, transition_hz=LP_TRANS_HZ)
print(f"FIR Parks-McClellan | HP N={len(hB_hp)} | BS N={len(hB_bs)} | LP N={len(hB_lp)}")

### 3.3. Método C: FIR baseado em FFT

Mesma resposta do Metodo A: o vetor `h` reaproveita o projeto Hamming.
A diferenca esta apenas na **forma de aplicar** (multiplicacao no dominio
da frequencia, via `scipy.signal.fftconvolve`).

In [ ]:
# kernels reutilizados de A
hC_hp, hC_bs, hC_lp = hA_hp, hA_bs, hA_lp
print("FIR via FFT reutiliza os kernels do metodo da janela.")

### 3.4. Método D: IIR Butterworth (ordem 4)

Aplicado em fase zero por `sosfiltfilt`. A ordem total efetiva e o dobro
(magnitude ao quadrado), e a fase resultante e zero.

In [ ]:
sosD_hp = iir.design_highpass_sos(FS, HP_CUT_HZ, order=4)
sosD_bs = iir.design_bandstop_sos(FS, PLI_LO_HZ, PLI_HI_HZ, order=4)
sosD_lp = iir.design_lowpass_sos(FS, LP_CUT_HZ, order=4)
print("IIR Butterworth ordem 4 em cada estagio (HP, BS, LP).")

### 3.5. Respostas em frequência

Aqui, optamos por fazer uma verificacao visual, em que cada metodo deve atender as bandas de rejeicao prescritas. A linha tracejada marca os cortes nominais.

In [ ]:
def plot_h_grid(filters, title):
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
    for ax, (label, h_or_sos, xlim, vline) in zip(axes, filters):
        if isinstance(h_or_sos, np.ndarray) and h_or_sos.ndim == 1:
            plot_filter_response(ax, FS, h=h_or_sos, title=label, xlim=xlim)
        else:
            plot_filter_response(ax, FS, sos=h_or_sos, title=label, xlim=xlim)
        for v in vline:
            ax.axvline(v, color="grey", ls="--", lw=0.6)
        ax.set_ylim(-120, 5)
    plt.suptitle(title, y=1.02); plt.tight_layout(); plt.show()

plot_h_grid([
    (f"A. HP @ {HP_CUT_HZ} Hz", hA_hp, (0, 5), [HP_CUT_HZ]),
    (f"A. BS {PLI_LO_HZ}-{PLI_HI_HZ} Hz", hA_bs, (50, 70), [PLI_LO_HZ, PLI_HI_HZ]),
    (f"A. LP @ {LP_CUT_HZ} Hz", hA_lp, (0, 80), [LP_CUT_HZ]),
], "Metodo A - FIR pelo metodo da janela (Hamming)")

plot_h_grid([
    (f"B. HP @ {HP_CUT_HZ} Hz", hB_hp, (0, 5), [HP_CUT_HZ]),
    (f"B. BS {PLI_LO_HZ}-{PLI_HI_HZ} Hz", hB_bs, (50, 70), [PLI_LO_HZ, PLI_HI_HZ]),
    (f"B. LP @ {LP_CUT_HZ} Hz", hB_lp, (0, 80), [LP_CUT_HZ]),
], "Metodo B - FIR por Parks-McClellan (equiripple)")

plot_h_grid([
    (f"D. HP @ {HP_CUT_HZ} Hz", sosD_hp, (0, 5), [HP_CUT_HZ]),
    (f"D. BS {PLI_LO_HZ}-{PLI_HI_HZ} Hz", sosD_bs, (50, 70), [PLI_LO_HZ, PLI_HI_HZ]),
    (f"D. LP @ {LP_CUT_HZ} Hz", sosD_lp, (0, 80), [LP_CUT_HZ]),
], "Metodo D - IIR Butterworth (ordem 4)")

## 4. Aplicação da cadeia HP → BS → LP

Cada método é aplicado no mesmo sinal, na mesma ordem, em fase zero
(`filtfilt` para FIR, `sosfiltfilt` para IIR, `fftconvolve` para o método
FFT).

In [ ]:
def chain_fir_filtfilt(x, h_hp, h_bs, h_lp):
    y = fir.apply_filtfilt(h_hp, x)
    y = fir.apply_filtfilt(h_bs, y)
    y = fir.apply_filtfilt(h_lp, y)
    return y

def chain_fir_fft(x, h_hp, h_bs, h_lp):
    y = fir.apply_fft(h_hp, x)
    y = fir.apply_fft(h_bs, y)
    y = fir.apply_fft(h_lp, y)
    return y

def chain_iir_sosfiltfilt(x, sos_hp, sos_bs, sos_lp):
    y = iir.apply_sosfiltfilt(sos_hp, x)
    y = iir.apply_sosfiltfilt(sos_bs, y)
    y = iir.apply_sosfiltfilt(sos_lp, y)
    return y

yA = chain_fir_filtfilt(x, hA_hp, hA_bs, hA_lp)
yB = chain_fir_filtfilt(x, hB_hp, hB_bs, hB_lp)
yC = chain_fir_fft(x, hC_hp, hC_bs, hC_lp)
yD = chain_iir_sosfiltfilt(x, sosD_hp, sosD_bs, sosD_lp)

results = {
    "A. FIR janela (filtfilt)": yA,
    "B. FIR Parks-McClellan (filtfilt)": yB,
    "C. FIR via FFT (fftconvolve)": yC,
    "D. IIR Butterworth (sosfiltfilt)": yD,
}
print("Cadeia aplicada para os quatro metodos.")

## 5. Comparação no domínio do tempo

In [ ]:
fig, axes = plt.subplots(len(results), 1, figsize=(11, 2.6 * len(results)), sharex=True)
for ax, (name, y) in zip(axes, results.items()):
    plot_overlay(ax, t, x, y, title=name, filt_label="filtrado")
axes[-1].set_xlabel("tempo (s)")
plt.tight_layout(); plt.show()

## 6. Comparação no domínio da frequência

PSD por Welch. A queda de potência fora da banda passante deve coincidir
com as bandas de rejeição projetadas (coerência filtro/PSD, critério da
Seção 7.1 do PDF).

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.0))
plot_psd(ax, {"cru": x, **results}, FS, title="PSD - sinal cru e quatro metodos", xlim=(0, 90))
for v in (HP_CUT_HZ, PLI_LO_HZ, PLI_HI_HZ, LP_CUT_HZ):
    ax.axvline(v, color="grey", ls="--", lw=0.6)
plt.tight_layout(); plt.show()

## 7. Linearidade de fase (atraso de grupo)

Atraso de grupo dos três estágios FIR (janela e Parks-McClellan) e do IIR
em SOS. Para FIR Tipo I e III a fase é exatamente linear: o atraso de
grupo é constante. Para o IIR sem `filtfilt`, há variação na banda passante.

In [ ]:
def gd_fir(h, fs, npoints=4096):
    w, gd = sp_signal.group_delay((h, [1.0]), w=npoints, fs=fs)
    return w, gd

def gd_sos(sos, fs, npoints=4096):
    b, a = sp_signal.sos2tf(sos)
    w, gd = sp_signal.group_delay((b, a), w=npoints, fs=fs)
    return w, gd

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, (label, h_a, h_b, sos_d, xlim) in zip(axes, [
    ("HP", hA_hp, hB_hp, sosD_hp, (0, 5)),
    ("BS", hA_bs, hB_bs, sosD_bs, (50, 70)),
    ("LP", hA_lp, hB_lp, sosD_lp, (0, 80)),
]):
    for name, h in [("A. janela", h_a), ("B. Parks-McClellan", h_b)]:
        w, gd = gd_fir(h, FS); ax.plot(w, gd, lw=0.9, label=name)
    w, gd = gd_sos(sos_d, FS); ax.plot(w, gd, lw=0.9, label="D. IIR (sem filtfilt)")
    ax.set_title(f"Atraso de grupo - estagio {label}")
    ax.set_xlabel("frequencia (Hz)"); ax.set_ylabel("amostras")
    ax.set_xlim(xlim); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 8. Verificação numérica: A e C produzem a mesma saída

Como C usa o mesmo kernel de A aplicado por convolução rápida, as saídas
devem ser numericamente próximas. Aqui replicamos o forward-backward via
FFT para isolar a equivalência com `filtfilt`.

In [ ]:
def fft_filtfilt(h, x):
    y = fir.apply_fft(h, x)
    y = fir.apply_fft(h, y[::-1])[::-1]
    return y

yA_eq = fft_filtfilt(hA_lp, x)
yA_re = fir.apply_filtfilt(hA_lp, x)
diff = np.max(np.abs(yA_eq - yA_re))
print(f"max |fftconvolve forward-backward - filtfilt| = {diff:.3e}")

## 9. Métricas consolidadas (Seção 7.1 do PDF)

| Métrica | Critério atendido |
|---------|-------------------|
| `redução_BW_dB`, `redução_PLI_dB`, `redução_EMG_dB` | redução de ruído nas bandas alvo |
| `preservação_passband_dB` | preservação morfológica em 1 a 30 Hz |
| `picos_R_RMS_ms`, `picos_R_max_ms` | estabilidade temporal de eventos |
| `gd_var_amostras` | linearidade de fase (variação do atraso de grupo na banda passante) |

In [ ]:
def gd_variation_passband(stages_eval, fs, band=PASSBAND, npoints=4096):
    total = 0.0
    for kind, obj in stages_eval:
        if kind == "fir":
            w, gd = sp_signal.group_delay((obj, [1.0]), w=npoints, fs=fs)
        else:
            b, a = sp_signal.sos2tf(obj)
            w, gd = sp_signal.group_delay((b, a), w=npoints, fs=fs)
        m = (w >= band[0]) & (w <= band[1])
        total += float(np.nanmax(gd[m]) - np.nanmin(gd[m]))
    return total

stages = {
    "A. FIR janela (filtfilt)": [("fir", hA_hp), ("fir", hA_bs), ("fir", hA_lp)],
    "B. FIR Parks-McClellan (filtfilt)": [("fir", hB_hp), ("fir", hB_bs), ("fir", hB_lp)],
    "C. FIR via FFT (fftconvolve)": [("fir", hC_hp), ("fir", hC_bs), ("fir", hC_lp)],
    "D. IIR Butterworth (sosfiltfilt)": [("sos", sosD_hp), ("sos", sosD_bs), ("sos", sosD_lp)],
}

rows = []
for name, y in results.items():
    rp = M.r_peak_shift_ms(ANN_LOCAL, y, FS)
    rows.append({
        "metodo": name,
        "reducao_BW_dB": M.power_reduction_db(x, y, FS, *BAND_BW),
        "reducao_PLI_dB": M.power_reduction_db(x, y, FS, *BAND_PLI),
        "reducao_EMG_dB": M.power_reduction_db(x, y, FS, *BAND_EMG),
        "preservacao_passband_dB": M.passband_change_db(x, y, FS, *PASSBAND),
        "picos_R_RMS_ms": rp["rms_ms"],
        "picos_R_max_ms": rp["max_ms"],
        "gd_var_amostras": gd_variation_passband(stages[name], FS),
    })

df = pd.DataFrame(rows).round(3)
display(df)

OUT_CSV = PROCESSED_DATA_DIR / "etapa01_v2_metrics.csv"
df.to_csv(OUT_CSV, index=False)
print(f"Tabela exportada: {OUT_CSV}")

## 10. Discussão

- **A vs C:** mesma resposta em frequência. Diferenças residuais vêm de
  `filtfilt` (fase zero, magnitude ao quadrado) versus `fftconvolve`
  (passe único, magnitude simples). O ganho de C é exclusivamente
  computacional.
- **A vs B:** mesmas bandas. Parks-McClellan atinge especificação
  comparável com `N` menor (ondulação uniforme nas bandas de rejeição),
  ao preço de fronteiras mais "duras".
- **D (IIR):** ordem total muito menor (4 por estágio), com atenuação
  comparável após `sosfiltfilt`. A penalidade aparece na linearidade de
  fase **antes** do `filtfilt` (coluna `gd_var_amostras` separa os FIR
  exatamente lineares dos IIR).
- **Estabilidade temporal:** em batimentos normais (registo 100),
  `picos_R_RMS_ms` deve ficar pequeno para todos os métodos (FIR de fase
  linear com `filtfilt` tipicamente abaixo de 5 ms). Diferenças residuais
  refletem transientes de borda e a magnitude ao quadrado de `filtfilt`.

A teoria de cada ponto está em
[`docs/etapa_01_denoising_v2.md`](../docs/etapa_01_denoising_v2.md).

## 11. Espectrograma (STFT) antes e depois do denoising

O espectrograma resolve o sinal no tempo **e** na frequência ao mesmo tempo.
A Transformada de Fourier de Curta Duração (STFT, do inglês *Short-Time Fourier
Transform*) divide o sinal em janelas sobrepostas, aplica a FFT em cada uma e
empilha os espectros ao longo do eixo temporal. O resultado é uma imagem
frequência x tempo onde a cor indica a potência.

Comparamos o sinal cru com a saída do Método A (FIR janela, filtfilt), que é
a técnica principal. O objetivo é verificar que o denoising **remove energia
nas bandas alvo** (abaixo de 0,5 Hz, em torno de 60 Hz, acima de 40 Hz)
**sem criar artefatos no interior da banda passante**.

In [ ]:
# Parametros do espectrograma: janela de 0,5 s com 90% de sobreposicao
STFT_WINDOW_SEC = 0.5
STFT_OVERLAP = 0.9

f_raw, t_raw, S_raw = gab.stft_spectrogram(x, FS, window_sec=STFT_WINDOW_SEC, overlap=STFT_OVERLAP)
f_filt, t_filt, S_filt = gab.stft_spectrogram(yA, FS, window_sec=STFT_WINDOW_SEC, overlap=STFT_OVERLAP)

vmin = max(S_raw.min(), -80)
vmax = S_raw.max()

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, sharey=True)
for ax, S, titulo in zip(axes, [S_raw, S_filt], ["Sinal cru", "A. FIR janela (filtfilt)"]):
    img = ax.pcolormesh(t_raw, f_raw, S, shading="gouraud",
                        cmap="inferno", vmin=vmin, vmax=vmax)
    ax.set_ylabel("frequencia (Hz)")
    ax.set_title(f"Espectrograma (STFT) - {titulo}")
    ax.set_ylim(0, 90)
    for freq in (HP_CUT_HZ, PLI_LO_HZ, PLI_HI_HZ, LP_CUT_HZ):
        ax.axhline(freq, color="cyan", lw=0.7, ls="--", alpha=0.7)
    plt.colorbar(img, ax=ax, label="dB/Hz")
axes[-1].set_xlabel("tempo (s)")
plt.tight_layout()
plt.show()

**Como ler o espectrograma.** Cada coluna vertical é um espectro instantâneo.
Cores quentes (amarelo/branco) indicam alta potência; cores frias (preto/roxo)
indicam baixa potência. As linhas tracejadas marcam as fronteiras de corte dos
filtros. No sinal filtrado, a energia fora da banda passante (acima das linhas
de 0,5 Hz e abaixo de 40 Hz, e na faixa de 60 Hz) deve ser visivelmente atenuada.

## 12. Comparação de espectrogramas entre os quatro métodos

Exibimos os espectrogramas de todos os métodos lado a lado para verificar
que as atenuações são coerentes com as respostas em frequência projetadas.

In [ ]:
all_outputs = {"cru": x, **results}
n = len(all_outputs)
fig, axes = plt.subplots(n, 1, figsize=(12, 2.6 * n), sharex=True, sharey=True)
for ax, (nome, sig) in zip(axes, all_outputs.items()):
    _, _, S = gab.stft_spectrogram(sig, FS, window_sec=STFT_WINDOW_SEC, overlap=STFT_OVERLAP)
    img = ax.pcolormesh(t_raw, f_raw, S, shading="gouraud",
                        cmap="inferno", vmin=vmin, vmax=vmax)
    ax.set_ylabel("freq (Hz)"); ax.set_ylim(0, 90)
    ax.set_title(nome)
    for freq in (HP_CUT_HZ, PLI_LO_HZ, PLI_HI_HZ, LP_CUT_HZ):
        ax.axhline(freq, color="cyan", lw=0.7, ls="--", alpha=0.7)
    plt.colorbar(img, ax=ax, label="dB/Hz")
axes[-1].set_xlabel("tempo (s)")
plt.tight_layout()
plt.show()

## 13. Filtros de Gabor 1D

Um filtro de Gabor 1D é uma onda senoidal modulada por um envelope gaussiano:

```
g(t, f0) = envelope(t) * cos(2*pi*f0*t)
```

O envelope concentra a energia do filtro em torno de t=0, o que confere
**localização simultânea no tempo e na frequência**: o filtro "olha" para
o conteúdo em torno de f0 em uma janela temporal de largura proporcional
a sigma.

**Parâmetro sigma.** Usamos `sigma = n_cycles / (2*pi*f0)`, com `n_cycles = 3`.
Isso garante que o filtro contenha sempre o mesmo número de oscilações dentro
do envelope, independentemente da frequência (banco Q-constante). A largura
espectral efetiva é `Δf ≈ f0 / n_cycles`: quanto maior f0, mais larga a
banda do filtro em Hz, mas sempre a mesma fração relativa de f0.

### 13.1. Visualização dos kernels de Gabor

Exibimos o kernel (parte real e envelope) para três frequências representativas.

In [ ]:
# frequencias centrais do banco de Gabor: do limite inferior ao superior da banda de ECG
GABOR_FREQS = np.array([1.0, 5.0, 10.0, 20.0, 30.0, 40.0])
N_CYCLES = 3.0

fig, axes = plt.subplots(1, 3, figsize=(13, 3.0))
for ax, f0 in zip(axes, [1.0, 10.0, 30.0]):
    t_k, h_real, _ = gab.gabor_kernel(f0, FS, n_cycles=N_CYCLES)
    envelope = np.exp(-(t_k ** 2) / (2 * (N_CYCLES / (2 * np.pi * f0)) ** 2))
    ax.plot(t_k, h_real, color="#1f77b4", lw=0.9, label="parte real")
    ax.plot(t_k, envelope, color="#d62728", lw=0.8, ls="--", label="envelope")
    ax.plot(t_k, -envelope, color="#d62728", lw=0.8, ls="--")
    ax.set_title(f"Gabor @ {f0} Hz (sigma={N_CYCLES/(2*np.pi*f0)*1000:.1f} ms)")
    ax.set_xlabel("tempo (s)"); ax.set_ylabel("amplitude"); ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Observe** que em 1 Hz o kernel é muito largo no tempo (sigma grande),
enquanto em 30 Hz é compacto. Isso reflete o compromisso de Heisenberg:
boa resolução em frequência implica resolução temporal ruim, e vice-versa.
Para ECG, frequências altas (bordas do QRS) exigem boa resolução temporal,
enquanto frequências baixas (onda T, P) podem ser medidas com janela mais larga.

### 13.2. Escalograma de Gabor (sinal cru vs filtrado)

In [ ]:
# Escalograma: energia de Gabor para cada frequencia do banco ao longo do tempo
sc_raw = gab.gabor_scalogram(x, FS, GABOR_FREQS, n_cycles=N_CYCLES)
sc_filt = gab.gabor_scalogram(yA, FS, GABOR_FREQS, n_cycles=N_CYCLES)

# Converte para dB (energia relativa)
def to_db(E):
    return 10.0 * np.log10(E + 1e-20)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, sharey=True)
for ax, sc, titulo in zip(axes, [sc_raw, sc_filt], ["Sinal cru", "A. FIR janela (filtfilt)"]):
    img = ax.pcolormesh(t, GABOR_FREQS, to_db(sc), shading="gouraud", cmap="viridis")
    ax.set_ylabel("frequencia central (Hz)")
    ax.set_title(f"Escalograma de Gabor - {titulo}")
    ax.set_yticks(GABOR_FREQS)
    plt.colorbar(img, ax=ax, label="energia (dB)")
axes[-1].set_xlabel("tempo (s)")
plt.tight_layout()
plt.show()

**Como ler o escalograma.** Cada linha horizontal corresponde à energia do
filtro de Gabor naquela frequência ao longo do tempo. Picos verticais (energia
alta em muitas frequências ao mesmo tempo) correspondem ao complexo QRS, que
é um evento de alta energia e banda larga. A onda T aparece como energia
concentrada em frequências mais baixas (1 a 10 Hz).

No sinal filtrado, o escalograma deve mostrar:
- ausência de energia em frequências fora da banda passante (acima de 40 Hz);
- manutenção da estrutura temporal dos picos QRS (posição e forma);
- possível leve redução na energia de frequências de borda (1 Hz) pelo efeito
  do passa-alta.

### 13.3. Comparação entre métodos no escalograma

In [ ]:
# Comparar os quatro metodos no escalograma: diferenca em relacao ao cru
fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharex=True, sharey=True)
axes_flat = axes.ravel()
for ax, (nome, yi) in zip(axes_flat, results.items()):
    sc_i = gab.gabor_scalogram(yi, FS, GABOR_FREQS, n_cycles=N_CYCLES)
    diff_db = to_db(sc_raw) - to_db(sc_i)  # positivo = reducao de energia
    img = ax.pcolormesh(t, GABOR_FREQS, diff_db, shading="gouraud",
                        cmap="RdBu_r", vmin=-20, vmax=20)
    ax.set_title(f"Reducao de energia - {nome}")
    ax.set_ylabel("freq (Hz)"); ax.set_yticks(GABOR_FREQS)
    plt.colorbar(img, ax=ax, label="dB (positivo = atenuou)")
axes_flat[2].set_xlabel("tempo (s)")
axes_flat[3].set_xlabel("tempo (s)")
plt.suptitle("Diferenca de energia Gabor: cru - filtrado (dB)", y=1.01)
plt.tight_layout()
plt.show()

**Como ler o mapa de diferença.** Tons de vermelho indicam redução de
energia pelo filtro (ruído removido); tons de azul indicam aumento de energia
(artefato introduzido pelo filtro). Um método ideal mostra vermelho apenas nas
bandas alvo (abaixo de 1 Hz, em torno de 30 a 40 Hz) e branco (neutro) na
banda passante ao longo de todo o sinal.

A métrica final da tabela consolidada (seção 9) complementa esta visualização:
o escalograma revela a dimensão **temporal** da atenuação, enquanto a PSD
revela a dimensão **global espectral**.